<a href="https://colab.research.google.com/github/misbahhassan6400/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

### Finding 1: Model versus rule baseline

The reported reference result shows that Precision@50 increased from approximately 0.26 for the rule-based baseline to approximately 0.74 for the Random Forest model.

My methodology question is: How exactly was the positive label defined, and was it created from a future outcome window that was fully separate from the input features? I would also check whether Precision@50 was measured on a held-out or grouped test set. Without that detail, the improvement is measured on the reported sample, but it may not support a broader claim about performance on unseen clients.

### Finding 2: CTR varies by ranking position

The reported weighted CTR comparison shows approximately 0.49% for top-three results, 0.35% for page-one results, and 0.04% for deeper-ranking results.

My methodology question is: How were the position buckets and weighted CTR calculated, and how many impressions contributed to each bucket? I would also check whether the comparison was made across an independent time window or validation sample, because traffic mix, client mix, and small denominators could affect the observed differences.

These questions are constructive checks on label provenance, aggregation, sample size, and validation design. They do not reject the findings; they define what evidence is needed before extending them beyond the reported data.

In [ ]:
!pip -q install duckdb huggingface_hub pandas scikit-learn scipy

import duckdb
import pandas as pd
import numpy as np
import json
import os

from google.colab import userdata

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN missing. Colab Secrets mein access ON karo.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

files = con.sql("""
SELECT file
FROM glob('hf://datasets/FlyRank/internship-warehouse/**/*.parquet')
""").df()

feb_matches = files[
    files["file"].astype(str).str.contains("2026-02", regex=False)
]

mar_matches = files[
    files["file"].astype(str).str.contains("2026-03", regex=False)
]

FEB = feb_matches.iloc[0]["file"]
MAR = mar_matches.iloc[0]["file"]

print("Setup complete")
print("FEB:", FEB)
print("MAR:", MAR)

Setup complete
FEB: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet
MAR: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet


## 2. My model under an honest split (before/after)

The before condition uses a random row split. This can place content from the same client in both train and test, so it may give an optimistic estimate.

The after condition uses a grouped split by client. All rows from one client stay in only one split, which better tests transfer to unseen clients. I keep the same features, Random Forest method, random seed, and metrics for both conditions.

In [ ]:
model_df = con.sql(f"""
WITH feb AS (
    SELECT
        client_hash_id,
        content_hash_id,

        COALESCE(
            SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE), 0
        ) AS gsc_impressions,

        COALESCE(
            SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE), 0
        ) AS gsc_clicks,

        AVG(gsc_avg_position)
            FILTER (
                WHERE gsc_data_available IS TRUE
                AND gsc_avg_position IS NOT NULL
            ) AS gsc_avg_position,

        COALESCE(
            SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE), 0
        ) AS ga4_sessions,

        COALESCE(
            SUM(ga4_engaged_sessions)
            FILTER (WHERE ga4_data_available IS TRUE), 0
        ) AS ga4_engaged_sessions

    FROM read_parquet('{FEB}')
    WHERE month = '2026-02'
    GROUP BY client_hash_id, content_hash_id
),

mar AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(ga4_sessions)
            FILTER (WHERE ga4_data_available IS TRUE)
            AS march_ga4_sessions

    FROM read_parquet('{MAR}')
    WHERE month = '2026-03'
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    feb.*,
    mar.march_ga4_sessions
FROM feb
INNER JOIN mar
    USING (client_hash_id, content_hash_id)
WHERE mar.march_ga4_sessions IS NOT NULL
""").df()

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

print("model_df created:", model_df.shape)
display(model_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

model_df created: (80877, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_ga4_sessions
0,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,17.806923,13.0,0.0,3.0
1,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,7.852945,1.0,0.0,1.0
2,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,7.965842,0.0,0.0,1.0
3,client_e547b89c05043229,content_c6dbda992ad84127,2868.0,3.0,17.993308,4.0,0.0,2.0
4,client_e547b89c05043229,content_cb6bc37251e57efa,688.0,0.0,15.972425,2.0,0.0,2.0


In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import mean_absolute_error
from scipy.stats import spearmanr

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = model_df[feature_cols].copy()
y = model_df["march_ga4_sessions"].astype(float)
groups = model_df["client_hash_id"].fillna("MISSING_CLIENT")

assert len(feature_cols) == 5
assert "march_ga4_sessions" not in feature_cols

print("Rows:", len(model_df))
print("Features:", feature_cols)

Rows: 80877
Features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'ga4_engaged_sessions']


In [ ]:
def build_model():
    rf_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=150,
            min_samples_leaf=10,
            random_state=42,
            n_jobs=-1
        ))
    ])

    return TransformedTargetRegressor(
        regressor=rf_pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

def spearman_metric(actual, predicted):
    result = spearmanr(actual, predicted)
    return float(result.statistic)

def top_decile_recall(actual, predicted):
    k = max(1, int(np.ceil(len(actual) * 0.10)))

    actual_top = set(np.argsort(actual)[-k:])
    predicted_top = set(np.argsort(predicted)[-k:])

    return len(actual_top & predicted_top) / k

def evaluate_split(train_idx, test_idx, split_name):
    model = build_model()

    model.fit(
        X.iloc[train_idx],
        y.iloc[train_idx]
    )

    predictions = model.predict(X.iloc[test_idx])
    predictions = np.maximum(predictions, 0)

    actual = y.iloc[test_idx].to_numpy()

    return {
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_clients": groups.iloc[train_idx].nunique(),
        "test_clients": groups.iloc[test_idx].nunique(),
        "spearman_rho": spearman_metric(actual, predictions),
        "mae": mean_absolute_error(actual, predictions),
        "top_decile_recall": top_decile_recall(actual, predictions)
    }, model, predictions

In [ ]:
all_indices = np.arange(len(model_df))

random_train_idx, random_test_idx = train_test_split(
    all_indices,
    test_size=0.20,
    random_state=42
)

random_result, random_model, random_predictions = evaluate_split(
    random_train_idx,
    random_test_idx,
    "Before: random row split"
)

print(random_result)

{'split': 'Before: random row split', 'train_rows': 64701, 'test_rows': 16176, 'train_clients': 36, 'test_clients': 35, 'spearman_rho': 0.6112872555432391, 'mae': 10.874615519290897, 'top_decile_recall': 0.6285537700865266}


In [ ]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

assert len(
    set(groups.iloc[group_train_idx])
    & set(groups.iloc[group_test_idx])
) == 0

group_result, group_model, group_predictions = evaluate_split(
    group_train_idx,
    group_test_idx,
    "After: grouped by client"
)

print(group_result)
print("Client overlap: 0")

{'split': 'After: grouped by client', 'train_rows': 66273, 'test_rows': 14604, 'train_clients': 28, 'test_clients': 8, 'spearman_rho': 0.5885719714298703, 'mae': 8.253650081984524, 'top_decile_recall': 0.649555099247091}
Client overlap: 0


In [ ]:
comparison = pd.DataFrame([
    random_result,
    group_result
])

display(comparison)

print(
    "Spearman change:",
    round(
        group_result["spearman_rho"]
        - random_result["spearman_rho"],
        4
    )
)

print(
    "Top-decile recall change:",
    round(
        group_result["top_decile_recall"]
        - random_result["top_decile_recall"],
        4
    )
)

,split,train_rows,test_rows,train_clients,test_clients,spearman_rho,mae,top_decile_recall
0,Before: random row split,64701,16176,36,35,0.611287,10.874616,0.628554
1,After: grouped by client,66273,14604,28,8,0.588572,8.253650,0.649555


Spearman change: -0.0227
Top-decile recall change: 0.021


The grouped-by-client result is the more honest estimate for transfer to unseen clients. The random split is treated as the before condition because related rows from the same client may appear in both partitions. The grouped result is measured under a stricter design, so any performance decrease is informative rather than a failure.

## 3. Leakage audit

The final feature set contains only February measurements. March GA4 sessions is the target and is not included in X. Client and content identifiers are used for grouping and auditing, not as model features. Product flags, future-window fields, URLs, and client names are excluded.

In [ ]:
final_feature_names = list(X.columns)

print("Final model features:")
for col in final_feature_names:
    print("-", col)

forbidden_terms = [
    "march",
    "label",
    "future",
    "outcome",
    "target",
    "flag",
    "url",
    "name"
]

leakage_name_matches = [
    col for col in final_feature_names
    if any(term in col.lower() for term in forbidden_terms)
]

print("Forbidden feature-name matches:", leakage_name_matches)

assert final_feature_names == feature_cols
assert "march_ga4_sessions" not in final_feature_names
assert leakage_name_matches == []

print("Leakage audit: PASS")
print("Feature window: February 2026")
print("Label window: March 2026")
print("March label used as feature: NO")
print("Product flags used: NO")

Final model features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions
Forbidden feature-name matches: []
Leakage audit: PASS
Feature window: February 2026
Label window: March 2026
March label used as feature: NO
Product flags used: NO


In [ ]:
failure_examples = model_df.iloc[group_test_idx].copy()

failure_examples["prediction"] = group_predictions
failure_examples["absolute_error"] = abs(
    failure_examples["march_ga4_sessions"]
    - failure_examples["prediction"]
)

failure_examples = failure_examples.sort_values(
    "absolute_error",
    ascending=False
)

display(
    failure_examples[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "ga4_sessions",
            "ga4_engaged_sessions",
            "march_ga4_sessions",
            "prediction",
            "absolute_error"
        ]
    ].head(10)
)


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,march_ga4_sessions,prediction,absolute_error
20085,94852.0,1123.0,2.559384,577.0,36.0,2730.0,582.845365,2147.154635
22702,44487.0,291.0,3.452090,293.0,22.0,191.0,540.421452,349.421452
22624,40080.0,260.0,2.514103,303.0,17.0,194.0,540.421452,346.421452
41315,24105.0,431.0,4.143416,412.0,31.0,521.0,224.445854,296.554146
60438,142215.0,1605.0,1.882000,395.0,14.0,293.0,577.291438,284.291438
60567,28237.0,162.0,3.141285,102.0,4.0,465.0,185.441706,279.558294
22317,30485.0,49.0,4.849475,36.0,0.0,354.0,78.020462,275.979538
26298,63169.0,165.0,3.149420,147.0,8.0,64.0,318.007711,254.007711
20631,119854.0,490.0,2.594817,450.0,30.0,816.0,582.845365,233.154635
63518,40374.0,412.0,2.717006,496.0,15.0,358.0,576.414746,218.414746


The largest errors are real held-out examples from clients not seen during training. They show where the model is uncertain, especially when traffic is heavy-tailed, Search Console values are missing, or March behavior differs from February patterns. These examples limit the model to directional decision-support rather than automatic action.

## 4. Claim rewrite

Original claim:

"The Random Forest predicts future content performance accurately and beats the baseline."

Safer rewrite:

"On this measured February-to-March sample, the Random Forest produced the reported ranking metrics under the selected split. The grouped-client result is a directional estimate for unseen-client transfer, not proof of causal impact, universal accuracy, or guaranteed improvement over the baseline."

## Self-check

- Two real findings from the research paper are named.
- One constructive methodology question is written for each finding.
- Random row split and grouped-client split are compared.
- Train and test clients do not overlap in the honest split.
- The same features, model, seed, and metrics are used in both conditions.
- March GA4 sessions is not included as a feature.
- A leakage audit is visible and passes.
- Real held-out failure examples are shown.
- My strongest claim is rewritten using observed, measured, directional, and decision-support language.
- No client names, URLs, private queries, or raw data files are included.
- The notebook runs with Runtime → Run all without errors.
- The notebook is committed under `work/notebooks/w06_validation_audit.ipynb`.